In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

client = OpenAI()

In [2]:
def llm(prompt):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text


llm("Hey, what's up?")

'Not much—just here and ready to help. What’s on your mind?'

In [3]:
question = "I just discovered the course. Can I join now?"
print(llm(question))

Yes—usually you can join even if the course has already started, but it depends on the course policy and whether there’s still room.

A few things to check:
- **Enrollment deadline:** some courses allow late registration
- **Missed material:** you may need to catch up on earlier lessons
- **Instructor approval:** some courses require permission to join late

If you want, I can help you draft a short message to the instructor asking if it’s still possible to enroll.


In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.
"""

prompt = f"""
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

print(llm(prompt))

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
courses_raw = requests.get(docs_url).json()
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [6]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

len(documents)

1350

In [7]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [8]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [9]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [10]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )


search_results = search(question)
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [11]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [12]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


print(build_context(search_results))

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [13]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [15]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [16]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [17]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [18]:
from ingest import load_faq_data
from sqlitesearch import TextSearchIndex

documents = load_faq_data()
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    sqlite_index.add(doc)

sqlite_index.close()
len(docs_llm)

85

In [19]:
from rag_helper import RAGBase

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

print(sqlite_index.count())

assistant = RAGBase(
    index=sqlite_index,
    llm_client=client,
)

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

85
Yes, you can still join the course after it started. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [20]:
import json


def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    return index.search(query, num_results=5, boost_dict=boost_dict, filter_dict=filter_dict)


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query text to look up in the course FAQ."}
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [21]:
messages = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

response = client.responses.create(model="gpt-5.4-mini", input=messages, tools=[search_tool])
response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course late discovered can I join now enrollment registration eligibility"}', call_id='call_1KGJoDrgTUr51MJ4ABByvcFT', name='search', type='function_call', id='fc_0225736043437cf3006a398c62f8dc819cb34a30fd8a74c85e', namespace=None, status='completed')]

In [22]:
call = response.output[0]
args = json.loads(call.arguments)
results = search(**args)

messages.extend(response.output)
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(results, indent=2),
})

response = client.responses.create(model="gpt-5.4-mini", input=messages, tools=[search_tool])
response.output_text

'Yes — you can still join the course and start learning anytime.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


def make_call(call):
    args = json.loads(call.arguments)
    if call.name == "search":
        result = search(**args)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, indent=2),
    }

In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    last_answer = ""
    it = 1
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = client.responses.create(model=model, input=messages, tools=[search_tool])
        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                messages.append(make_call(item))
                has_function_calls = True
            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(last_answer)

        it += 1
        if not has_function_calls:
            break

    return last_answer

In [25]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run Ollama locally install run local course FAQ"}
iteration #2...
function_call: search {"query":"Ollama run llama3 localhost 11434 Python client ollama serve course FAQ"}
iteration #3...
ASSISTANT:
If you mean **Ollama**, here’s the quick way to run it locally:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download and install the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['mess

'If you mean **Ollama**, here’s the quick way to run it locally:\n\n1. **Install Ollama**\n   - macOS: download from https://ollama.com/download and install the `.pkg`\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

In [27]:
result = runner.loop(prompt="How do I run Olama locally?", callback=callback)

-> Response received


-> Response received


-> Response received
